In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score
)
from sklearn.decomposition import PCA


# Load dataset
df = pd.read_csv('input.csv')


# Select numerical features for clustering
numerical_cols = [
    'ARG_Count', 'ASN_Count', 'ASP_Count', 'CYS_Count',
    'GLN_Count', 'GLU_Count', 'GLY_Count', 'HIS_Count',
    'MET_Count', 'SER_Count', 'THR_Count', 'TYR_Count',
    'VAL_Count', 'PHE_Count', 'ILE_Count',
    'LEU_Count', 'ALA_Count', 'LYS_Count', 'TRP_Count',
    'Comb-Atom_N', 'Comb-Atom_S', 'Comb-Atom_O',
    'Class_Nature_Non-polar', 'Class_Nature_Polar acidic',
    'Class_Nature_Polar basic', 'Class_Nature_Polar neutral',
    'Class_Nature_Polar O', 'Aromatic_Count',
    'Average_Isoelectric_Point', 'Average_Hydrophobicity'
]


# Handle missing values
df[numerical_cols] = df[numerical_cols].fillna(0)


# Add binary cysteine-presence feature
df['Cys_presence'] = (df['CYS_Count'] > 0).astype(int)


# Build feature matrix excluding Cys_presence for PCA
X_raw = df[numerical_cols].values


# Apply Yeo-Johnson transformation and standardization
pt = PowerTransformer(method='yeo-johnson')
X_trans = pt.fit_transform(X_raw)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_trans)


# Perform PCA while retaining 95% of the variance
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)


# Add weighted Cys_presence feature after PCA
weight_factor = 10

X_final = np.hstack([
    X_pca,
    (df[['Cys_presence']] * weight_factor).values
])


# Define clustering range and linkage methods
k_values = range(2, 36)

linkage_methods = {
    'Agglomerative - ward': 'ward',
    'Agglomerative - single': 'single',
    'Agglomerative - complete': 'complete',
    'Agglomerative - average': 'average'
}


# Calculate clustering evaluation metrics
metrics_results = []

for algorithm_name, linkage in linkage_methods.items():

    for k in k_values:

        clustering = AgglomerativeClustering(
            n_clusters=k,
            linkage=linkage
        )

        labels = clustering.fit_predict(X_final)

        metrics_results.append({
            'Algorithm': algorithm_name,
            'k': k,
            'Silhouette Score': silhouette_score(X_final, labels),
            'Davies-Bouldin Index': davies_bouldin_score(X_final, labels)
        })


metrics_df = pd.DataFrame(metrics_results)


# Define colors for Agglomerative clustering methods
custom_colors = {
    'Agglomerative - ward': 'orange',
    'Agglomerative - single': 'purple',
    'Agglomerative - complete': 'brown',
    'Agglomerative - average': 'green'
}


# Define plot information
metrics = [
    'Silhouette Score',
]

y_labels = [
    'Silhouette Score',
]

titles = [
    'Silhouette Score Comparison',
]

file_names = [
    'silhouette_score_comparison.png',
]


# Generate clustering evaluation plots
for metric, y_label, title, file_name in zip(
    metrics,
    y_labels,
    titles,
    file_names
):

    plt.figure(figsize=(8, 6))

    for algorithm_name, color in custom_colors.items():

        subset = metrics_df[
            metrics_df['Algorithm'] == algorithm_name
        ]

        plt.plot(
            subset['k'],
            subset[metric],
            marker='o',
            label=algorithm_name,
            color=color
        )

    plt.title(
        title,
        fontsize=22,
        fontweight='bold'
    )

    plt.xlabel(
        'Number of clusters (k)',
        fontsize=20,
        fontweight='bold'
    )

    plt.ylabel(
        y_label,
        fontsize=20,
        fontweight='bold'
    )

    plt.xticks(
        fontsize=18,
        fontweight='bold'
    )

    plt.yticks(
        fontsize=18,
        fontweight='bold'
    )

    plt.grid(
        True,
        linestyle='--',
        alpha=0.7
    )

    plt.savefig(
        file_name,
        dpi=1500,
        bbox_inches='tight'
    )

    plt.show()


# Create a separate legend image
plt.figure(figsize=(8, 6))

legend_handles = [
    plt.Line2D(
        [],
        [],
        color=color,
        marker='o',
        linestyle='',
        label=algorithm_name
    )
    for algorithm_name, color in custom_colors.items()
]

plt.legend(
    handles=legend_handles,
    loc='center',
    fontsize=14,
    ncol=len(custom_colors),
    frameon=False
)

plt.axis('off')

plt.savefig(
    'legend.png',
    dpi=1500,
    bbox_inches='tight'
)

plt.show()


print("Agglomerative clustering evaluation plots and legend saved.")

In [ ]:
# Run Ward Agglomerative Clustering for k = 2–39
k_values = range(2, 36)

sil_scores = []
db_scores = []

for k in k_values:

    model = AgglomerativeClustering(
        n_clusters=k,
        linkage='ward'
    )

    labels = model.fit_predict(X_final)

    sil_scores.append(
        silhouette_score(X_final, labels)
    )

    db_scores.append(
        davies_bouldin_score(X_final, labels)
    )


# Plot Silhouette Score
plt.figure(figsize=(10, 5))

plt.plot(
    k_values,
    sil_scores,
    marker='o',
    label='Silhouette Score'
)

plt.title('Silhouette Score vs. Number of Clusters (k)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


# Plot Davies–Bouldin Score
plt.figure(figsize=(10, 5))

plt.plot(
    k_values,
    db_scores,
    marker='^',
    label='Davies–Bouldin Score'
)

plt.title('Davies–Bouldin Score vs. Number of Clusters (k)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Davies–Bouldin Score (lower = better)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


# Summary of clustering evaluation metrics
df_metrics = pd.DataFrame({
    'k': k_values,
    'Silhouette': sil_scores,
    'Davies_Bouldin': db_scores
})

print(df_metrics)